In [7]:
%pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.1/463.1 kB 8.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.1/301.1 kB 22.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 MB 37.0 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import os
BASE_DIR = "/content/drive/MyDrive/MLMA_Dataset/data"
# Utility to join paths easily
def get_path(path):
    return os.path.join(BASE_DIR, path)

In [10]:
import numpy as np
import tensorflow as tf
import random as rn
import os
import matplotlib.pyplot as plt
%matplotlib inline
os.environ['PYTHONHASHSEED'] = '0'
np.random.seed(1)
rn.seed(1)
from keras import backend as K
tf.compat.v1.set_random_seed(1)
#sess = tf.compat.v1.Session(graph=tf.compat.v1.get_default_graph())
#K.set_session(sess)
import sys
from keras.models import Sequential
from keras.layers import LSTM, Dense, Activation, Dropout, Flatten
from keras.layers import Conv1D, MaxPooling1D, Add, Input, GlobalAveragePooling1D
from keras.models import Model
from keras.layers import BatchNormalization
from keras.optimizers import SGD
from keras.optimizers import RMSprop
import keras.regularizers
import scipy
import math
import sys
import pandas as pd
from scipy.ndimage.filters import gaussian_filter1d
from sklearn.metrics import mean_squared_error
from scipy.stats import linregress
from scipy import interpolate
from scipy import signal
import pickle
import collections
from keras.callbacks import LearningRateScheduler
from keras.callbacks import ModelCheckpoint
from keras.callbacks import TerminateOnNaN
from video_process_utils import *
import statsmodels.api as sm

/tmp/ipykernel_9258/2320229005.py:27: DeprecationWarning: Please import `gaussian_filter1d` from the `scipy.ndimage` namespace; the `scipy.ndimage.filters` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.ndimage.filters import gaussian_filter1d


In [11]:
target_col = 'KneeFlex_maxExtension'

In [12]:
alldata_processed =\
    pd.read_csv(get_path("processed/alldata_processed_with_dev_residual.csv") )
alldata_processed['videoid'] = alldata_processed['videoid'].apply(lambda x: int(x))
alldata_processed['target_count'] = alldata_processed.groupby('videoid')[target_col].transform(lambda x: x.count())

In [13]:
alldata_processed = alldata_processed[alldata_processed[target_col].notnull()]
alldata_processed = alldata_processed[alldata_processed['target_count'] == 2]
ids_nonmissing_target = set(alldata_processed['videoid'].unique())

In [14]:
datasplit_df = pd.read_csv(get_path("processed/train_test_valid_id_split.csv") )
datasplit_df['videoid'] = datasplit_df['videoid'].apply(lambda x: int(x))
all_ids = set(datasplit_df['videoid']).intersection(ids_nonmissing_target)
train_ids = set(datasplit_df[datasplit_df['dataset'] == 'train']['videoid']).intersection(ids_nonmissing_target)
validation_ids = set(datasplit_df[datasplit_df['dataset'] == 'validation']['videoid']).intersection(ids_nonmissing_target)
test_ids = set(datasplit_df[datasplit_df['dataset'] == 'test']['videoid']).intersection(ids_nonmissing_target)

In [15]:
with open(get_path("processed/all_processed_video_segments_doublesided.pickle"), 'rb') as handle:
    processed_video_segments = pickle.load(handle)

In [16]:
assert(np.sum(alldata_processed[target_col].isnull()) == 0)
alldata_processed

,Patient_ID,examid,side,HipFlex_IC,HipRot_mean,KneeFlex_meanStance,KneeFlex_maxExtension,dxmod,dxside,faq,...,height_interpolated,mass_buckets,height_buckets,age_buckets,mass_interpolated2,age_truncated2,height_interpolated2,predicted_SEMLS,SEMLS_dev_residual,target_count
0,3466,6167,R,65.103000,42.098059,49.812311,43.406333,Diplegia,Bilateral,6,...,137.795,55.0,130.0,20.0,31.329767,4.000000,189.87462,0.085150,-0.421888,2
1,3466,6167,L,71.249667,-0.169098,52.859310,47.368000,Diplegia,Bilateral,6,...,137.795,55.0,130.0,20.0,31.329767,4.000000,189.87462,0.085150,-0.421888,2
2,3523,10742,R,44.691429,-14.877223,22.957506,9.374398,Hemiplegia,Left,10,...,164.000,65.0,160.0,20.0,47.196900,4.000000,268.96000,0.064930,-0.366425,2
3,3523,10742,L,47.951407,8.404332,17.942818,9.420285,Hemiplegia,Left,10,...,164.000,65.0,160.0,20.0,47.196900,4.000000,268.96000,0.064930,-0.366425,2
4,3561,8676,R,45.436579,-9.705642,29.875777,24.770209,Diplegia,Bilateral,0,...,175.000,65.0,170.0,20.0,48.580900,4.000000,306.25000,0.058696,-0.347818,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4419,10117,14849,L,43.525260,-3.446997,11.446938,7.087655,Diplegia,Bilateral,5,...,157.000,80.0,150.0,20.0,112.996900,4.000000,246.49000,0.043852,-0.299474,2
4420,10121,14857,R,33.502858,1.984192,17.656262,9.921491,Hemiplegia,Right,10,...,144.500,30.0,140.0,11.0,9.486400,1.431465,208.80250,0.202391,-0.672513,2
4421,10121,14857,L,42.464181,-0.131036,21.262660,9.046386,Hemiplegia,Right,10,...,144.500,30.0,140.0,11.0,9.486400,1.431465,208.80250,0.202391,-0.672513,2
4422,10162,14940,R,39.679867,-9.121854,17.941415,7.365353,Hemiplegia,Left,9,...,149.400,45.0,140.0,12.0,22.562500,1.616033,223.20360,0.178429,-0.626956,2


In [17]:
x_columns_left = [2*LANK,2*LANK+1,2*LKNE,2*LKNE+1,
        2*LHIP,2*LHIP+1,2*LBTO,2*LBTO+1,50,52,54,56]
x_columns_right = [2*RANK,2*RANK+1,2*RKNE,2*RKNE+1,
        2*RHIP,2*RHIP+1,2*RBTO,2*RBTO+1,51,53,55,57]

target_dict_L = {}
target_dict_R = {}
for i in range(len(alldata_processed)):
    row = alldata_processed.iloc[i]
    if row['side'] == 'L':
        target_dict_L[row['videoid']] = row[target_col]
    if row['side'] == 'R':
        target_dict_R[row['videoid']] = row[target_col]

In [18]:
X = [t[2] for t in processed_video_segments if t[0] in all_ids]
X = np.stack(X)
X = np.concatenate([X[:,:,x_columns_left],X[:,:,x_columns_right]])

In [19]:
X_train = [t[2] for t in processed_video_segments if t[0] in train_ids]
X_train = np.stack(X_train)
X_train = np.concatenate([X_train[:,:,x_columns_left],X_train[:,:,x_columns_right]])

In [20]:
X_validation = [t[2] for t in processed_video_segments if t[0] in validation_ids]
X_validation = np.stack(X_validation)
X_validation = np.concatenate([X_validation[:,:,x_columns_left],X_validation[:,:,x_columns_right]])

In [21]:
y_train_L = [target_dict_L[t[0]] for t in processed_video_segments if t[0] in train_ids]
y_train_R = [target_dict_R[t[0]] for t in processed_video_segments if t[0] in train_ids]
y_train = np.array(y_train_L + y_train_R)

In [22]:
y_validation_L = [target_dict_L[t[0]] for t in processed_video_segments if t[0] in validation_ids]
y_validation_R = [target_dict_R[t[0]] for t in processed_video_segments if t[0] in validation_ids]
y_validation = np.array(y_validation_L + y_validation_R)

In [23]:
y_L = [target_dict_L[t[0]] for t in processed_video_segments if t[0] in all_ids]
y_R = [target_dict_R[t[0]] for t in processed_video_segments if t[0] in all_ids]
y = np.array(y_L + y_R)

In [24]:
videoid_count_dict = collections.Counter(np.array([t[0] for t in processed_video_segments]))

In [25]:
train_videoid_weights = [1./videoid_count_dict[t[0]] for t in processed_video_segments if t[0] in train_ids]
train_videoid_weights = train_videoid_weights + train_videoid_weights
train_videoid_weights = np.array(train_videoid_weights).reshape(-1,1)
validation_videoid_weights = [1./videoid_count_dict[t[0]] for t in processed_video_segments if t[0] in validation_ids]
validation_videoid_weights = validation_videoid_weights + validation_videoid_weights
validation_videoid_weights = np.array(validation_videoid_weights).reshape(-1,1)

In [26]:
target_min = np.min(y_train,axis=0)
target_range = np.max(y_train,axis=0) - np.min(y_train,axis=0)
print(target_min, target_range)

-24.4896329410691 109.8919662744024


In [27]:
y_train_scaled = ((y_train-target_min)/target_range).reshape(-1,1)
y_validation_scaled = ((y_validation-target_min)/target_range).reshape(-1,1)
y_validation_scaled = np.hstack([y_validation_scaled,validation_videoid_weights])
y_train_scaled = np.hstack([y_train_scaled,train_videoid_weights])
#c_i_factor is just a constant initially introduced to make the loss function more interpretable, but
#is probably not necessary if the model were to be retrained
c_i_factor = np.mean(np.vstack([train_videoid_weights,validation_videoid_weights]))

In [28]:
vid_length = 124

In [29]:
def step_decay(initial_lrate, epochs_drop, drop_factor):
    def step_decay_fcn(epoch):
        return initial_lrate * math.pow(drop_factor, math.floor((1 + epoch) / epochs_drop))
    return step_decay_fcn

epochs_drop, drop_factor = (10, 0.8)
initial_lrate = 0.001
dropout_amount = 0.5
last_layer_dim = 10
filter_length = 8
conv_dim = 32
l2_lambda = 10**(-3.5)

def w_mse(weights):
    def loss(y_true, y_pred):
        return K.mean(K.sum(K.square(y_true - y_pred) * weights, axis=1) * tf.reshape(y_true[:, -1], (-1, 1))) / c_i_factor
    return loss

# We don't want to optimize for the column counting video occurrences, but
# they are included in the target so we can use that column for the loss function
weights = [1.0, 0]

# Fixed epoch budget of 100 that empirically seems to be sufficient
n_epochs = 100

mse_opt = w_mse(weights)

# This is just for monitoring
mse_metric = w_mse(target_range**2 * np.array(weights))

K.clear_session()

# ---------------------------------------------------------------------------
# Dilated TCN (Temporal Convolutional Network) — replaces the original CNN
# ---------------------------------------------------------------------------
# Each residual block applies two dilated causal Conv1D layers with the same
# dilation rate, adds a 1x1 projection of the input (skip connection), and
# applies dropout. Stacking blocks with exponentially increasing dilation
# rates (1, 2, 4, 8, ...) gives a large receptive field without pooling.
# ---------------------------------------------------------------------------

from keras.layers import Add, Lambda

def tcn_residual_block(x, conv_dim, filter_length, dilation_rate, dropout_amount, l2_lambda=None):
    """Single TCN residual block with dilated causal convolutions."""
    reg = keras.regularizers.l2(l2_lambda) if l2_lambda else None

    # Two dilated causal Conv1D layers
    out = Conv1D(conv_dim, filter_length,
                 padding='causal',
                 dilation_rate=dilation_rate,
                 kernel_regularizer=reg)(x)
    out = BatchNormalization()(out)
    out = Activation('relu')(out)
    out = Dropout(dropout_amount)(out)

    out = Conv1D(conv_dim, filter_length,
                 padding='causal',
                 dilation_rate=dilation_rate,
                 kernel_regularizer=reg)(out)
    out = BatchNormalization()(out)
    out = Activation('relu')(out)
    out = Dropout(dropout_amount)(out)

    # 1x1 projection for residual connection (match channel dimension)
    if x.shape[-1] != conv_dim:
        x = Conv1D(conv_dim, 1)(x)

    out = Add()([out, x])
    return out


from keras.models import Model
from keras.layers import Input, GlobalAveragePooling1D

inputs = Input(shape=(vid_length, X_train.shape[2]))

# First projection to conv_dim channels (no dilation, no regularizer)
x = Conv1D(conv_dim, filter_length, padding='causal')(inputs)
x = BatchNormalization()(x)
x = Activation('relu')(x)

# Stack of residual blocks with exponentially increasing dilation rates
dilation_rates = [1, 2, 4, 8, 16]
for d in dilation_rates:
    x = tcn_residual_block(
        x, conv_dim, filter_length, dilation_rate=d,
        dropout_amount=dropout_amount,
        l2_lambda=l2_lambda if d > 1 else None   # no reg on first block
    )

# Global average pooling collapses the time dimension
x = GlobalAveragePooling1D()(x)
x = Dense(last_layer_dim, activation='relu')(x)
outputs = Dense(2, activation='linear')(x)

model = Model(inputs=inputs, outputs=outputs)
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 124, 12)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 124, 32)   │      3,104 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 124, 32)   │        128 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 124, 32)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 124, 32)   │      8,224 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 124, 32)   │        128 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 124, 32)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 124, 32)   │          0 │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 124, 32)   │      8,224 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 124, 32)   │        128 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 124, 32)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 124, 32)   │          0 │ activation_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 124, 32)   │          0 │ dropout_1[0][0],  │
│                     │                   │            │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 124, 32)   │      8,224 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 124, 32)   │        128 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 124, 32)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 124, 32)   │          0 │ activation_3[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 124, 32)   │      8,224 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 124, 32)   │        128 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 87,104 (340.25 KB)

 Trainable params: 86,400 (337.50 KB)

 Non-trainable params: 704 (2.75 KB)

In [30]:
checkpoint_folder = get_path("checkpoints/cnn_checkpoints_%s" % (target_col))

In [ ]:
train_model = True

if not os.path.exists(checkpoint_folder):
    os.makedirs(checkpoint_folder)

filepath=checkpoint_folder+"/weights-{epoch:02d}-{val_loss_1:.4f}.keras"
if train_model:

    # Redefine w_mse using tf.math functions and add a name argument
    def w_mse(weights, name=None):
        def loss_fn(y_true, y_pred):
            return tf.math.reduce_mean(tf.math.reduce_sum(tf.math.square(y_true-y_pred)*weights,axis=1)*tf.reshape(y_true[:,-1],(-1,1)))/c_i_factor
        if name:
            loss_fn.__name__ = name # Assign the desired name
        return loss_fn

    # Redefine mse_opt and mse_metric to use the updated w_mse with specific names
    mse_opt = w_mse(weights, name='weighted_mse_loss') # Give the primary loss a distinct name
    mse_metric = w_mse(target_range**2*np.array(weights), name='loss_1') # Name the metric 'loss_1'

    opt = RMSprop(learning_rate=0.0,rho=0.9, epsilon=1e-08)
    model.compile(loss=mse_opt,metrics=[mse_metric],
                  optimizer=opt)


    checkpoint = \
        ModelCheckpoint(filepath, monitor='val_loss_1', verbose=1, save_best_only=False, save_weights_only=False, mode='min', save_freq='epoch')

    lr = LearningRateScheduler(step_decay(initial_lrate,epochs_drop,drop_factor))

    history = model.fit(X_train, y_train_scaled,callbacks=[checkpoint,lr,TerminateOnNaN()],
              validation_data=(X_validation,y_validation_scaled),
              batch_size=32, epochs=n_epochs,shuffle=True)

Epoch 1/100
1277/1277 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step - loss: 0.2346 - loss_1: 2214.9444
Epoch 1: saving model to /content/drive/MyDrive/MLMA_Dataset/data/checkpoints/cnn_checkpoints_KneeFlex_maxExtension/weights-01-403.2006.keras

Epoch 1: finished saving model to /content/drive/MyDrive/MLMA_Dataset/data/checkpoints/cnn_checkpoints_KneeFlex_maxExtension/weights-01-403.2006.keras
1277/1277 ━━━━━━━━━━━━━━━━━━━━ 192s 143ms/step - loss: 0.0688 - loss_1: 475.3944 - val_loss: 0.0398 - val_loss_1: 403.2006 - learning_rate: 0.0010
Epoch 2/100
1277/1277 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - loss: 0.0144 - loss_1: 119.8645
Epoch 2: saving model to /content/drive/MyDrive/MLMA_Dataset/data/checkpoints/cnn_checkpoints_KneeFlex_maxExtension/weights-02-167.7103.keras

Epoch 2: finished saving model to /content/drive/MyDrive/MLMA_Dataset/data/checkpoints/cnn_checkpoints_KneeFlex_maxExtension/weights-02-167.7103.keras
1277/1277 ━━━━━━━━━━━━━━━━━━━━ 179s 140ms/step - loss: 0.0124 - loss_1: 109.1956 -

In [ ]:
def undo_scaling(y,target_range,target_min):
    return y*target_range+target_min

In [ ]:
weight_files = os.listdir(checkpoint_folder)
weight_files_df = pd.DataFrame(weight_files,columns=['filename'])
weight_files_df['num'] = weight_files_df['filename'].apply(lambda x: int(x.split('-')[1]))
weight_files_df.sort_values(by='num',ascending=True,inplace=True)

In [ ]:
#todo: change so we also have a X,y for all_ids
#then for each of 100 epochs, generate the predictions of the model for each epoch
#this dataframe can then be saved, and a file called "get_best_epoch" can be used to generate the best one
#(video_id,target,predictions,dataset)

In [ ]:
def predict_and_aggregate(y,X,ids,model,target_col):
    df = pd.DataFrame(y,columns=[target_col])
    side_col = ['L' if i < len(df)/2 else 'R' for i in range(len(df))]
    target_col_pred = target_col + "_pred"
    videoids = [t[0] for t in processed_video_segments if t[0] in ids]
    videoids = videoids + videoids
    df["videoid"] = np.array(videoids)
    df['side'] = side_col
    preds = model.predict(X)
    df[target_col_pred] = undo_scaling(preds[:,0],target_range,target_min)
    df["count"] = 1
    df = df.groupby(['videoid','side'],as_index=False).agg({target_col_pred:np.mean,'count':np.sum,target_col:np.mean})
    df['ones'] = 1
    return df

In [ ]:
video_ids = [t[0] for t in processed_video_segments if t[0] in all_ids]
video_ids = video_ids + video_ids
predictions_df = pd.DataFrame(video_ids,columns=['videoid'])
side_col = ['L' if i < len(predictions_df)/2 else 'R' for i in range(len(predictions_df))]
predictions_df[target_col] = y
predictions_df['side'] = side_col
predictions_df = predictions_df.merge(right=datasplit_df[['videoid','dataset']],on=['videoid'],how='left')

In [ ]:
for i in range(0,len(weight_files_df)):
    weight_file = weight_files_df['filename'].iloc[i]
    print(weight_file)
    model.load_weights(checkpoint_folder + "/%s" % (weight_file))
    preds = model.predict(X)
    predictions_df["%s_pred_%s" % (target_col,i)] = undo_scaling(preds[:,0],target_range,target_min)

In [ ]:
predictions_df.groupby(['videoid','side','dataset'],as_index=False).mean().to_csv(get_path("predictions/cnn_%s_predictions_all_epochs.csv" % (target_col)),index=False)

In [ ]:
# Save best models
maps = {
    "KneeFlex_maxExtension": get_path("checkpoints/cnn_checkpoints_KneeFlex_maxExtension/weights-48-82.5100.keras"),
}
for col in maps.keys():
    model_folder_path = get_path("models/%s_best.keras" % (col))
    model.load_weights(maps[col])
    model.save(model_folder_path)